# Content Moderation, from the inside

A moderation system decides three things about a post: does it stay up, does it
come down, or does a person look at it. This notebook builds the decision, in
standard library Python, and measures it.

It is self contained. No API key, no installs, no clone. Everything below runs
in order and computes its own numbers.

The full project this comes from is a React and FastAPI application with six
agents on a LangGraph state machine. What that adds is scale and a dashboard.
What it does not add is a different answer to the questions here.

## 1. The layer that runs before any model

Every post hits a keyword detector first. It is fast, free, deterministic, and
it produces the toxicity number the rest of the pipeline reasons about. The
language model never computes that score. It is handed the score and asked
about context.

That makes this small function more load bearing than it looks. Here it is,
written the way the project originally had it.

In [ ]:
PROFANITY = ["damn", "hell", "crap", "ass", "bastard", "piss"]
INSULTS = ["idiot", "stupid", "moron", "loser", "pathetic", "worthless",
           "trash", "scum", "rat", "clueless", "useless"]
THREATS = ["kill", "murder", "attack", "hurt", "destroy", "die", "beat",
           "off", "massacre", "hack", "take out", "burn"]


def score_substring(text):
    """The original: `word in text_lower` is a SUBSTRING test, not a word test."""
    low = text.lower()
    p = sum(1 for w in PROFANITY if w in low)
    i = sum(1 for w in INSULTS if w in low)
    t = sum(1 for w in THREATS if w in low)
    score = min(p * 0.15, 0.3) + min(i * 0.2, 0.4) + min(t * 0.3, 0.6)
    return min(score, 1.0), (p, i, t)


GREETING = "Hello everyone, I am grateful for the offer. Let us discuss the strategy at the office."
INSULT = "You are all worthless idiots and should be removed from this site."

for label, text in (("greeting", GREETING), ("insult", INSULT)):
    score, counts = score_substring(text)
    print(f"{label:9} {score:.2f}   profanity/insult/threat counts = {counts}")

The greeting scores higher than the insult.

`Hello` contains `hell`. `grateful` contains `rat`. `offer` contains `off`,
which is on the threat list because "off" is slang for kill. Three lists fire
on a sentence with nothing wrong in it, and only one fires on the insult.

This is not a contrived example. It was the live behaviour, and the greeting is
the one that got flagged for human review.

In [ ]:
def why(text):
    low = text.lower()
    hits = [(w, kind) for kind, lst in (("profanity", PROFANITY), ("insult", INSULTS), ("threat", THREATS))
            for w in lst if w in low]
    for w, kind in hits:
        where = next(word for word in text.split() if w in word.lower())
        print(f"  {kind:9} list entry {w!r:12} matched inside {where!r}")

why(GREETING)

## 2. Word boundaries, and the trap in the fix

The fix is to match words rather than fragments. That is one regex, but the
obvious version of it introduces a second bug immediately.

In [ ]:
import re
from functools import lru_cache


@lru_cache(maxsize=64)
def matcher(terms, plurals=True):
    ordered = sorted(terms, key=len, reverse=True)
    body = "|".join(re.escape(t) for t in ordered)
    tail = r"(?:e?s)?" if plurals else ""
    return re.compile(r"\b(?:" + body + r")" + tail + r"\b")


def score_boundary(text, plurals=True):
    low = text.lower()
    counts = []
    for lst in (PROFANITY, INSULTS, THREATS):
        counts.append(len(set(matcher(tuple(lst), plurals).findall(low))))
    p, i, t = counts
    score = min(p * 0.15, 0.3) + min(i * 0.2, 0.4) + min(t * 0.3, 0.6)
    return min(score, 1.0), (p, i, t)


print("naive word boundaries, no plural handling")
for label, text in (("greeting", GREETING), ("insult", INSULT)):
    score, counts = score_boundary(text, plurals=False)
    print(f"  {label:9} {score:.2f}  {counts}")

The greeting is fixed. But look at the insult: it fell too.

`\bidiot\b` does not match `idiots`, because the plural has a word character
exactly where the boundary needs to be. The score of a real insult halved, and
nothing raised an error. A fix that quietly weakens detection is worse than the
bug it replaced, because now you believe the problem is solved.

In [ ]:
print(f"{'sentence':<12} {'substring':>10} {'boundary':>10} {'+plurals':>10}")
for label, text in (("greeting", GREETING), ("insult", INSULT)):
    a, _ = score_substring(text)
    b, _ = score_boundary(text, plurals=False)
    c, _ = score_boundary(text, plurals=True)
    print(f"{label:<12} {a:>10.2f} {b:>10.2f} {c:>10.2f}")

print()
print("The last column is the only one where the insult outranks the greeting")
print("AND detection is not weakened. The fix is two changes, not one.")

## 3. What word boundaries cannot fix

Boundaries fix a mechanical bug. They do nothing about words that are genuinely
ambiguous, and that residue is the entire argument for putting a model behind
this layer.

In [ ]:
CASES = [
    ("greeting",      GREETING),
    ("school report", "The class passed the assessment and the teacher was pleased."),
    ("shop talk",     "We should hack together a demo and take out a subscription."),
    ("history",       "This documentary about the massacre at Wounded Knee is essential viewing."),
    ("real insult",   INSULT),
    ("real threat",   "I will find you and hurt you badly, you pathetic loser."),
]

for label, text in CASES:
    score, _ = score_boundary(text)
    flag = "FLAG" if score >= 0.3 else "ok  "
    print(f"{flag} {score:.2f}  {label:<14} {text[:56]}")

Shop talk outranks a real insult. "hack" and "take out" are both genuinely on
the threat list and both have an everyday meaning, and a history lesson about a
massacre contains the word massacre.

No amount of list curation fixes this, because the difference is not in the
words. It is in whether the words are being used or mentioned. That is the one
thing a model can do here that a list cannot.

## 4. The graph, and the three ways out

Posts do not walk a pipeline end to end. Each stage decides what runs next, and
three of those decisions end the run early. Here is the routing, small enough to
read in one go.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Account:
    name: str
    violations: int = 0
    reputation: float = 0.75
    verified: bool = False
    followers: int = 120


@dataclass
class State:
    text: str
    account: Account
    score: float = 0.0
    status: str = "submitted"
    agents: list = field(default_factory=list)
    calls: int = 0
    hitl_reasons: list = field(default_factory=list)


def content_analysis(s):
    s.calls += 2                      # topic extraction, then an assessment
    s.score, _ = score_boundary(s.text)
    s.agents.append("content")
    # Anything the first pass already considers bad goes straight to a person.
    s.status = "flagged" if s.score >= 0.35 else "continue"
    return s


def toxicity(s):
    s.calls += 1
    s.agents.append("toxicity")
    return s


def policy(s):
    s.calls += 1
    s.agents.append("policy")
    return s


def react(s):
    s.calls += 1
    s.agents.append("react")
    if s.account.verified or s.account.followers >= 10_000:
        s.hitl_reasons.append("high_profile_user")
    if s.score >= 0.7 and s.account.violations == 0:
        s.hitl_reasons.append("first_offense_severe")
    if s.hitl_reasons:
        s.status = "pending_human_review"
    return s


def reputation(s):
    s.calls += 1
    s.agents.append("reputation")
    risk = s.account.violations * 0.2 + (1 - s.account.reputation) * 0.5
    if risk > 0.7 or (s.account.violations >= 3 and s.score >= 0.15):
        s.status = "enforce"
    else:
        s.status = "approved"
    return s


def enforce(s):
    s.calls += 1
    s.agents.append("enforce")
    s.status = "removed"
    return s


def run_full(text, account):
    s = State(text, account)
    s = content_analysis(s)
    if s.status == "flagged":
        return s                       # exit one: straight to the moderator queue
    s = toxicity(s)
    s = policy(s)
    s = react(s)
    if s.status == "pending_human_review":
        return s                       # exit two: the graph pauses for a human
    s = reputation(s)
    if s.status == "approved":
        return s                       # exit three: done, nobody looked
    return enforce(s)

## 5. The same sentence, three answers

Now the part that looks like a bug and is not. Three accounts post identical
text.

In [ ]:
SAME = "You are being an idiot about this."

ACCOUNTS = [
    Account("newcomer"),
    Account("repeat offender", violations=4, reputation=0.25),
    Account("verified creator", verified=True, followers=50_000),
]

print(f"text: {SAME!r}")
print(f"keyword score: {score_boundary(SAME)[0]:.2f} for all three\n")
print(f"{'account':<18} {'agents':<38} {'calls':>5}  status")
for acct in ACCOUNTS:
    s = run_full(SAME, acct)
    print(f"{acct.name:<18} {' > '.join(s.agents):<46} {s.calls:>5}  {s.status}")

Nothing about the text was reinterpreted. The account history changed what the
correct response was, which is what moderation is actually for. A first offence
and a fourth offence are not the same event even when the words match.

The third row is a different mechanism. A large verified account is escalated
regardless of score, because the cost of being wrong is asymmetric: removing a
big creator's post in error is a public incident, and leaving it up for twenty
minutes while a person looks is not.

## 6. The fast path, and what the shortcut costs

Short comments can skip the whole thing. One model call decides. This is the
most consequential setting in the system, and it is an environment variable.

In [ ]:
def run_fast(text, account):
    """One call. Never loads reputation, never evaluates a HITL trigger."""
    s = State(text, account)
    s.calls = 1
    s.agents = ["fast"]
    s.score, _ = score_boundary(text)
    s.status = "removed" if s.score >= 0.6 else "approved"
    return s


full_calls = fast_calls = 0
disagreements = 0
print(f"{'account':<18} {'full path':<26} {'fast path':<22}")
for acct in ACCOUNTS:
    f, q = run_full(SAME, acct), run_fast(SAME, acct)
    full_calls += f.calls
    fast_calls += q.calls
    same = f.status == q.status
    disagreements += 0 if same else 1
    mark = "" if same else "  <-- differs"

    def cell(state):
        unit = "call" if state.calls == 1 else "calls"
        return f"{state.status} ({state.calls} {unit})"

    print(f"{acct.name:<18} {cell(f):<26} {cell(q):<22}{mark}")

print()
print(f"model calls: {full_calls} full, {fast_calls} fast "
      f"({(1 - fast_calls / full_calls) * 100:.0f}% fewer)")
print(f"outcomes changed: {disagreements} of {len(ACCOUNTS)}")

The saving is real and so is the loss.

The fast path cannot suspend a repeat offender or escalate a large account,
because it never loads the data those decisions need. Routing short comments
here is a statement about which mistakes you are willing to make, and it belongs
in a policy document rather than a performance ticket.

In the full project, measured across five posts, the numbers are the same shape:
twenty-six model calls on the full pipeline against five on the fast path, with
two of the five outcomes changing.

## 7. Where the decisions really live

Almost everything consequential in this system is a number in a config file
rather than a piece of architecture. Here is the confidence threshold that
decides how much work reaches a human.

In [ ]:
import random

rng = random.Random(17)

# Mostly ordinary posts, because a real feed is mostly ordinary. Using the
# earlier CASES list directly would make four posts in six already toxic, and
# the follower threshold would then be swamped by the content flag.
BENIGN = [t for lbl, t in CASES if score_boundary(t)[0] < 0.3]
TOXIC = [t for lbl, t in CASES if score_boundary(t)[0] >= 0.3]

POPULATION = []
for i in range(2000):
    text = rng.choice(TOXIC) if rng.random() < 0.08 else rng.choice(BENIGN)
    POPULATION.append((
        Account(f"user{i}",
                violations=rng.choice([0, 0, 0, 1, 2]),
                reputation=rng.uniform(0.4, 0.95),
                verified=rng.random() < 0.01,
                followers=int(10 ** rng.uniform(1, 5.3))),
        text,
    ))

print(f"{'threshold':>12} {'content-flagged':>16} {'escalated by size':>18} {'total to a human':>18}")
for threshold in (100_000, 10_000, 1_000):
    flagged = escalated = 0
    for acct, text in POPULATION:
        s = content_analysis(State(text, acct))
        if s.status == "flagged":
            flagged += 1
        elif acct.verified or acct.followers >= threshold:
            escalated += 1
    total = flagged + escalated
    print(f"{threshold:>12,} {flagged:>16} {escalated:>18} "
          f"{total:>13} ({total / len(POPULATION) * 100:.0f}%)")

The first column does not move, because it is content, not policy. The second
column is the dial.

Dropping the follower threshold from 100,000 to 1,000 leaves the detection
untouched and multiplies the review load, because follower counts are roughly
log-distributed and each factor of ten sweeps in far more accounts than the one
above it. That is a staffing budget wearing the costume of a constant.

The same is true of the confidence threshold, the fast-path length limit, and
the list of severities that always see a person. If you are auditing a
moderation system, read the config before the architecture.

## What to take away

- The keyword layer computes the number everything else reasons about, so its
  bugs are systemic rather than local. A substring test made a greeting score
  higher than an insult.
- A fix that silently weakens detection is worse than the bug. Word boundaries
  without plural handling halved the score of a real insult and raised nothing.
- Ambiguity is not fixable by list curation. Use against mention is the reason
  there is a model in the pipeline at all.
- The worst content should get the least machine work, because a human is going
  to decide it anyway.
- Identical text deserves different answers depending on the account, and that
  is a feature.
- The fast path buys latency with blindness. Price it as a policy decision.

The full project, with the six-agent LangGraph pipeline, the React dashboard,
the appeal workflow and the five guardrails, is on GitHub:

https://github.com/genieincodebottle/aiml-companion/tree/main/projects/agentic-ai/content-moderation-project

`python run.py demo` there runs the real graph with no API key.